# Store Sales Analysis
## Exploratory Data Analysis
* **Goal:** Understand the structure, quality and charachteristics of the dataset.

In [49]:
# Import libraries 

import pandas as pd 
from pathlib import Path
import matplotlib as plt
import seaborn as sns

In [50]:
# Define path to data folder

data_path = Path('../data')
data_path

PosixPath('../data')

In [51]:
# Find all csv in the folder

files = list(data_path.glob('*.csv'))
files

[PosixPath('../data/test.csv'),
 PosixPath('../data/train.csv'),
 PosixPath('../data/transactions.csv'),
 PosixPath('../data/oil.csv'),
 PosixPath('../data/holidays_events.csv'),
 PosixPath('../data/sample_submission.csv'),
 PosixPath('../data/stores.csv')]

In [52]:
# Create dictionary for all files

dataframes = {}

for file in files:
    name = file.stem
    dataframes[name] = pd.read_csv(file)

In [53]:
# Check dictionary

dataframes.keys()

dict_keys(['test', 'train', 'transactions', 'oil', 'holidays_events', 'sample_submission', 'stores'])

In [54]:
dataframes['train'].head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [55]:
# Check shape of all dataframes

for name, df in dataframes.items():
    print(name, df.shape)

test (28512, 5)
train (3000888, 6)
transactions (83488, 3)
oil (1218, 2)
holidays_events (350, 6)
sample_submission (28512, 2)
stores (54, 5)


In [56]:
# Check missing values for all dataframes

for name, df in dataframes.items():
    print('-'*30)
    print(name)
    print(df.isna().sum())

------------------------------
test
id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64
------------------------------
train
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64
------------------------------
transactions
date            0
store_nbr       0
transactions    0
dtype: int64
------------------------------
oil
date           0
dcoilwtico    43
dtype: int64
------------------------------
holidays_events
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64
------------------------------
sample_submission
id       0
sales    0
dtype: int64
------------------------------
stores
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64


In [57]:
# Inspect missing values in oil dataframe

dataframes['oil'][dataframes['oil']['dcoilwtico'].isna()]

,date,dcoilwtico
0,2013-01-01,NaN
14,2013-01-21,NaN
34,2013-02-18,NaN
63,2013-03-29,NaN
104,2013-05-27,NaN
132,2013-07-04,NaN
174,2013-09-02,NaN
237,2013-11-28,NaN
256,2013-12-25,NaN
261,2014-01-01,NaN


In [58]:
dataframes['oil'].head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


### Data Quality Check

The automated data quality check showed that most datasets contain no missing values. The `oil` dataset contains 43 missing values in `dcoilwtico`. These missing values occur on specific dates, mainly holidays and non-trading days, so they are not necessarily errors. Further investigation is needed before deciding how to handle them.

In [59]:
# Check dtype for all dataframes

for name, df in dataframes.items():
    print('-'*30)
    print(name)
    print(df.dtypes)
    print()

------------------------------
test
id              int64
date           object
store_nbr       int64
family         object
onpromotion     int64
dtype: object

------------------------------
train
id               int64
date            object
store_nbr        int64
family          object
sales          float64
onpromotion      int64
dtype: object

------------------------------
transactions
date            object
store_nbr        int64
transactions     int64
dtype: object

------------------------------
oil
date           object
dcoilwtico    float64
dtype: object

------------------------------
holidays_events
date           object
type           object
locale         object
locale_name    object
description    object
transferred      bool
dtype: object

------------------------------
sample_submission
id         int64
sales    float64
dtype: object

------------------------------
stores
store_nbr     int64
city         object
state        object
type         object
cluster       int

In [60]:
# Convert all date objects columns into datetime

for name, df in dataframes.items():
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])

In [61]:
# Check columns tolist

for name, df in dataframes.items():
    print('-'*30)
    print(name)
    print(df.columns.tolist())

------------------------------
test
['id', 'date', 'store_nbr', 'family', 'onpromotion']
------------------------------
train
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']
------------------------------
transactions
['date', 'store_nbr', 'transactions']
------------------------------
oil
['date', 'dcoilwtico']
------------------------------
holidays_events
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']
------------------------------
sample_submission
['id', 'sales']
------------------------------
stores
['store_nbr', 'city', 'state', 'type', 'cluster']


In [62]:
# Check nunique for stores_nbr
dataframes['stores']['store_nbr'].nunique()

54

In [63]:
# Check nunique for stores_nbr
dataframes['train']['store_nbr'].nunique()

54

In [64]:
# Check len of stres
len(dataframes['stores'])

54

In [65]:
# Check if there is information in train set which are not in store
set(dataframes['train']['store_nbr']) - set(dataframes['stores']['store_nbr'])

set()

In [66]:
set(dataframes['stores']['store_nbr']) - set(dataframes['train']['store_nbr'])

set()

In [67]:
# Merge train dataset with store dataset

train = dataframes['train'].merge(
    dataframes['stores'], 
    on='store_nbr',
    how='left'
)

In [68]:
# Check shape of new dataset
print(f'Rows:{train.shape[0]} -- Columns:{train.shape[1]}')

Rows:3000888 -- Columns:10


In [69]:
# Check head
train.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13


In [70]:
# Check missing values
train.isna().sum()

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
city           0
state          0
type           0
cluster        0
dtype: int64

In [71]:
# Check head of transactions df
dataframes['transactions'].head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [72]:
dataframes['transactions'].duplicated(
    subset=['date', 'store_nbr']
).sum()

np.int64(0)

In [73]:
train_keys = set(zip(train['date'], train['store_nbr']))
transaction_keys = set(
    zip(
        dataframes['transactions']['date'],
        dataframes['transactions']['store_nbr']
    )
)
len(train_keys-transaction_keys)

7448

In [74]:
missing_keys = train_keys - transaction_keys
list(missing_keys)[:10]

[(Timestamp('2014-06-25 00:00:00'), 29),
 (Timestamp('2014-03-15 00:00:00'), 21),
 (Timestamp('2014-01-01 00:00:00'), 12),
 (Timestamp('2013-12-30 00:00:00'), 22),
 (Timestamp('2013-02-18 00:00:00'), 21),
 (Timestamp('2014-09-30 00:00:00'), 22),
 (Timestamp('2013-05-10 00:00:00'), 52),
 (Timestamp('2013-10-29 00:00:00'), 29),
 (Timestamp('2014-05-10 00:00:00'), 22),
 (Timestamp('2015-09-20 00:00:00'), 52)]

In [75]:
missing_dates = [date for date, store in missing_keys]
pd.Series(missing_dates).value_counts().head(10)

2016-01-01    54
2016-01-03    54
2017-01-01    53
2015-01-01    53
2013-01-01    53
2014-01-01    52
2016-01-04    40
2016-01-02    18
2013-06-19    11
2014-05-23     8
Name: count, dtype: int64

In [76]:
pd.Series(missing_dates).dt.day_name().value_counts()

Sunday       1112
Tuesday      1072
Wednesday    1065
Friday       1061
Thursday     1058
Monday       1055
Saturday     1025
Name: count, dtype: int64

In [77]:
pd.Series(missing_dates).nunique()

1566

In [78]:
dataframes["transactions"]["date"].min(), dataframes["transactions"]["date"].max()

(Timestamp('2013-01-01 00:00:00'), Timestamp('2017-08-15 00:00:00'))

In [79]:
pd.Series(
    [store for date, store in missing_keys]
).value_counts().head(10)

52    1566
22    1013
42     964
21     936
29     810
20     775
53     517
36     133
18     118
24     107
Name: count, dtype: int64

In [80]:
# Merge with transaction
train = train.merge(
    dataframes['transactions'],
    on=['date', 'store_nbr'], 
    how='left'
)